In [7]:
import os
import ast
import pandas as pd
import numpy as np
import requests
from deepeval import evaluate
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase
from dotenv import load_dotenv, dotenv_values
from deepeval.models import AzureOpenAIModel

In [8]:
# LOADING INPUT QUESTIONS
goldens = pd.read_csv("../synthetic_data/goldens.csv")
data = goldens['input']
data = data[:3]

In [9]:
# CONNECTION TO CHATBOT ENDPOINT
endpoint = "https://hia-search-dev.azurewebsites.net/chat-dummy"
# loading variables from .env file
load_dotenv()
key = os.getenv("chatbot_key")
# making the HTTPS request to the chatbot endpoint (as client)
responses = []
for q in data:
  try:
    r = requests.post(params={"api_key": key, "include_context": True}, url=endpoint, json={"message": q})
  except requests.RequestException as e:
    print(f"Error: Failed to send request for question: {q}. Error: {e}")
    continue
  if r.status_code != 200:
    print(f"Error: Received status code {r.status_code} for question: {q}")
    continue
  response_dict = r.json()
  responses.append({'user_input' : q,
    'bot_output': response_dict['response'],
    'context': response_dict['context']})

# loading variables from .env file
load_dotenv(r"C:\Users\dari\Desktop\grad project 2026\HIA-eval-framework\.env")
subscription_key = os.getenv("azure_subscription_key")
api_version = "2024-12-01-preview"

# CUSTOM MODEL
endpoint = "https://510-ai-research.openai.azure.com/"
model = "gpt-4.1"
deployment = "gpt-4.1-students"
# loading variables from .env file
subscription_key = os.getenv("azure_subscription_key")
api_version = "2024-12-01-preview"
# model
custom_model = AzureOpenAIModel(
  model=deployment,
  api_key=subscription_key,
  azure_endpoint=endpoint,
  api_version=api_version,
  deployment_name=deployment
)

In [10]:
# TEST CASE CREATION
test_cases = []
for i in range(len(responses)):
    tc = LLMTestCase(
        input=responses[i]['user_input'],
        actual_output=responses[i]['bot_output'],
        retrieval_context=responses[i]['context'],
    )
    test_cases.append(tc)

# METRIC CALCULATION
metric = FaithfulnessMetric(
    threshold=0.5,
    include_reason=True,
    model=custom_model
)

In [11]:
evaluation_results = evaluate(test_cases=test_cases, metrics=[metric])

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...

c:\Users\dari\conda\envs\hia\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Faithfulness (score: 0.75, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The score is 0.75 because the actual output incorrectly claims that losing access to support organizations would prevent undocumented migrants from accessing medical care and children from attending school, while the retrieval context clarifies that these rights are guaranteed regardless of support organizations. The output is mostly faithful but contains notable inaccuracies., error: None)

For test case:

  - input: If UM lost access to all support orgs, what new risks might they face in NL?
  - actual output: If undocumented migrants (UM) lost access to all support organizations in the Netherlands, they might face new risks including:

- Homelessness or unsafe living conditions due to lack of shelter options.

- Inability to access medical care, leading to worsening health issues.

- Lack of legal assistance, increasing vulnerability to exploitation 

⚠ WARNING: No hyperparameters logged.
» ]8;id=615565;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 52.98s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [12]:
# OUTPUT AS DATAFRAME
results = []
for test_result in evaluation_results.test_results:
    row = {
        "input": test_result.input,
        "bot_output": test_result.actual_output,
        "retrieval_context": test_result.retrieval_context
    }
    for metric_data in test_result.metrics_data:
        row[metric_data.name] = metric_data.score
        row[metric_data.name + "_reason"] = metric_data.reason
    results.append(row)
results_df = pd.DataFrame(results)

In [13]:
print(results_df.columns.tolist())
print(results_df.shape)

['input', 'bot_output', 'retrieval_context', 'Faithfulness', 'Faithfulness_reason']

(3, 5)